# Data Engineering Skill File Generator Framework for Genie Code

## Overview
This framework generates comprehensive skill files for Databricks Data Engineering features, designed for use with Genie Code. It covers the complete spectrum of DE needs across development, testing, and production environments.

## Supported Data Engineering Features

### Pipeline & Data Processing
- **Lakeflow Spark Declarative Pipelines (SDP)** - Streaming tables, materialized views, CDC
- **Auto Loader** - Cloud storage ingestion with schema evolution
- **Change Data Capture (CDC)** - Upserts, deletes, SCD Type 1/2
- **Streaming Pipelines** - Real-time data processing
- **Batch Processing** - Scheduled data processing
- **Incremental Processing** - Efficient delta processing

### Data Quality & Architecture
- **Data Quality & Expectations** - Validation rules, data contracts
- **Medallion Architecture** - Bronze/Silver/Gold layer patterns
- **Delta Lake Optimization** - OPTIMIZE, Z-ORDER, VACUUM
- **Unity Catalog Governance** - Access control, lineage, tags

### Operations & Observability
- **Monitoring & Observability** - Metrics, logging, alerting
- **Testing Frameworks** - Unit tests, integration tests, data validation
- **Error Handling & Recovery** - Retry logic, dead letter queues
- **Performance Tuning** - Partitioning, caching, optimization

### Deployment & SDLC
- **CI/CD Deployment** - Asset bundles, automated deployment
- **SDLC Integration** - Dev/Test/Prod workflows
- **Environment Management** - Configuration, secrets, parameters

## Framework Components
1. **DE Feature Catalog** - Comprehensive metadata for all features
2. **Interactive Input System** - Widget-based user input collection
3. **Template Engine** - Dynamic skill file generation
4. **Output Generator** - SKILL.md formatted files with examples

## Usage
Run cells sequentially to:
1. Load dependencies and feature catalog
2. Configure your requirements via widgets
3. Generate professional skill files ready for Genie Code integration

In [0]:
# Import required libraries for skill file generation
import json
import os
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Any, Optional
import re

# For interactive widgets
try:
    from IPython.display import display, Markdown, HTML
except ImportError:
    pass

print("✓ All dependencies loaded successfully")
print(f"✓ Framework initialized at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"✓ Current workspace: {os.getcwd()}")

In [0]:
# Comprehensive Data Engineering Feature Catalog
# Updated for detailed Medallion sublayers and explicit ingestion, jobs, pipeline categories

DE_FEATURE_CATALOG = {
    # --- Medallion Layer Sublayers ---
    "bronze_layer": {
        "name": "Bronze Layer Pattern (Raw Ingestion)",
        "description": "Comprehensive guidance for implementing the bronze/raw data layer in Medallion architecture. This layer handles initial data ingestion, preserves data lineage, maintains immutable raw data, and provides the foundation for downstream silver and gold layers.",
        "category": "medallion_bronze",
        "use_cases": [
            "Initial landing of raw data from external sources (S3, ADLS, GCS, APIs)",
            "Immutable storage for compliance and audit requirements",
            "Late-arriving data and schema evolution handling",
            "Multi-source data consolidation with source tracking",
            "Historical data preservation for replay scenarios",
            "Raw data quality baseline establishment"
        ],
        "parameters": {
            "source_path": "string - Cloud storage path for raw data (s3://, abfss://, gs://)",
            "checkpoint_path": "string - Location for streaming checkpoint metadata",
            "schema_location": "string - Path for inferred schema storage and evolution",
            "file_format": "[json, csv, parquet, avro, xml] - Source file format",
            "ingestion_pattern": "[auto_loader, streaming, batch] - Data ingestion approach",
            "rescue_data_column": "boolean - Enable _rescued_data column for schema mismatch"
        },
        "code_templates": {
            "bronze_autoloader_json": """# Bronze Layer: Auto Loader for JSON files with schema evolution
import dlt
from pyspark.sql.functions import current_timestamp, input_file_name

@dlt.table(
    name="bronze_events_raw",
    comment="Raw event data ingested from cloud storage with Auto Loader",
    table_properties={
        "quality": "bronze",
        "pipelines.autoOptimize.managed": "true"
    }
)
def ingest_bronze_events():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "{checkpoint_path}/schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("rescuedDataColumn", "_rescued_data")
        .load("{source_path}")
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", input_file_name())
    )
""",
            "bronze_batch_csv": """# Bronze Layer: Batch ingestion for CSV files
import dlt
from pyspark.sql.functions import current_timestamp, input_file_name, lit, current_date

@dlt.table(
    name="bronze_customers_raw",
    comment="Raw customer data from CSV files",
    partition_cols=["ingestion_date"]
)
def ingest_bronze_customers():
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .load("{source_path}")
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("ingestion_date", current_date())
        .withColumn("source_file", input_file_name())
    )
""",
            "bronze_streaming_kafka": """# Bronze Layer: Streaming ingestion from Kafka
import dlt
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

@dlt.table(
    name="bronze_kafka_events",
    comment="Raw events streamed from Kafka topic"
)
def ingest_kafka_bronze():
    # Define expected schema
    event_schema = StructType([
        StructField("event_id", StringType(), True),
        StructField("user_id", StringType(), True),
        StructField("event_type", StringType(), True),
        StructField("timestamp", StringType(), True)
    ])
    
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "{kafka_brokers}")
        .option("subscribe", "{kafka_topic}")
        .option("startingOffsets", "latest")
        .option("failOnDataLoss", "false")
        .load()
        .select(
            col("key").cast("string").alias("kafka_key"),
            from_json(col("value").cast("string"), event_schema).alias("data"),
            col("topic"),
            col("partition"),
            col("offset"),
            col("timestamp").alias("kafka_timestamp")
        )
        .select("kafka_key", "data.*", "topic", "partition", "offset", "kafka_timestamp")
        .withColumn("ingestion_timestamp", current_timestamp())
    )
""",
            "bronze_with_metadata": """# Bronze Layer: Enhanced with comprehensive metadata tracking
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

@dlt.table(
    name="bronze_transactions_raw",
    comment="Raw transaction data with complete audit trail",
    table_properties={
        "delta.enableChangeDataFeed": "true",
        "delta.columnMapping.mode": "name"
    }
)
def ingest_bronze_transactions():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", "{checkpoint_path}/schema")
        .load("{source_path}")
        # Add comprehensive metadata
        .withColumn("bronze_id", expr("uuid()"))
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("ingestion_date", current_date())
        .withColumn("source_file", input_file_name())
        .withColumn("source_file_modification_time", col("_metadata.file_modification_time"))
        .withColumn("source_file_size", col("_metadata.file_size"))
        .withColumn("pipeline_id", lit(spark.conf.get("spark.databricks.pipelineId", "unknown")))
        .withColumn("pipeline_run_id", lit(spark.conf.get("spark.databricks.pipelineRunId", "unknown")))
    )
"""
        },
        "notebook_templates": {
            "complete_bronze_autoloader": {
                "name": "Complete Bronze Layer with Auto Loader",
                "description": "End-to-end bronze layer implementation using Auto Loader for JSON/CSV/Parquet files with schema evolution, monitoring, and best practices",
                "cells": [
                    {
                        "type": "markdown",
                        "content": """# Bronze Layer Implementation - Auto Loader Pattern

## Overview
This notebook implements a production-ready bronze layer using Databricks Auto Loader for incremental file ingestion from cloud storage.

## Key Features
* **Auto Loader** for scalable file discovery and ingestion
* **Schema Evolution** to handle source schema changes gracefully
* **Metadata Tracking** for lineage and debugging
* **Data Quality** baseline with rescued data column
* **Monitoring** queries for observability

## Prerequisites
* Unity Catalog enabled with catalog.schema created
* Cloud storage path with read permissions
* Checkpoint location with write permissions
"""
                    },
                    {
                        "type": "python",
                        "content": """# Configuration - Update these parameters for your environment

# Source configuration
source_path = "/databricks-datasets/retail-org/customers/"  # Update with your path
file_format = "csv"  # json, csv, parquet, avro

# Target configuration
catalog_name = "main"
schema_name = "bronze"
table_name = "customers_raw"

# Checkpoint configuration
checkpoint_path = f"/tmp/checkpoints/{catalog_name}/{schema_name}/{table_name}"
schema_location = f"{checkpoint_path}/schema"

# Display configuration
print("📋 Bronze Layer Configuration:")
print(f"  Source Path: {source_path}")
print(f"  File Format: {file_format}")
print(f"  Target Table: {catalog_name}.{schema_name}.{table_name}")
print(f"  Checkpoint: {checkpoint_path}")
"""
                    },
                    {
                        "type": "python",
                        "content": """# Create catalog and schema if they don't exist

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

print(f"✓ Catalog and schema ready: {catalog_name}.{schema_name}")
"""
                    },
                    {
                        "type": "python",
                        "content": """# Bronze Layer Ingestion with Auto Loader

from pyspark.sql.functions import current_timestamp, input_file_name, col, current_date

# Read stream using Auto Loader
df_bronze = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", file_format)
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # Handle schema changes
    .option("rescuedDataColumn", "_rescued_data")  # Capture malformed records
    .load(source_path)
    # Add metadata columns
    .withColumn("bronze_ingestion_timestamp", current_timestamp())
    .withColumn("bronze_ingestion_date", current_date())
    .withColumn("bronze_source_file", input_file_name())
)

# Write to bronze table
query = (
    df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")  # Allow schema evolution
    .partitionBy("bronze_ingestion_date")  # Partition for performance
    .trigger(processingTime="5 minutes")  # Micro-batch every 5 minutes
    .toTable(f"{catalog_name}.{schema_name}.{table_name}")
)

print(f"✓ Bronze ingestion started: {catalog_name}.{schema_name}.{table_name}")
print(f"  Stream ID: {query.id}")
print(f"  Status: {query.status}")
"""
                    },
                    {
                        "type": "markdown",
                        "content": """## Data Quality Checks

Monitor the bronze layer for:
* Record counts and ingestion rate
* Schema evolution events
* Rescued data (malformed records)
* Ingestion lag
"""
                    },
                    {
                        "type": "sql",
                        "content": """-- Monitor bronze table - Record counts by ingestion date
SELECT 
    bronze_ingestion_date,
    COUNT(*) as record_count,
    COUNT(DISTINCT bronze_source_file) as file_count,
    MIN(bronze_ingestion_timestamp) as first_ingestion,
    MAX(bronze_ingestion_timestamp) as last_ingestion
FROM {catalog_name}.{schema_name}.{table_name}
GROUP BY bronze_ingestion_date
ORDER BY bronze_ingestion_date DESC
LIMIT 10
"""
                    },
                    {
                        "type": "sql",
                        "content": """-- Check for rescued data (malformed records)
SELECT 
    bronze_ingestion_date,
    bronze_source_file,
    _rescued_data,
    COUNT(*) as malformed_count
FROM {catalog_name}.{schema_name}.{table_name}
WHERE _rescued_data IS NOT NULL
GROUP BY bronze_ingestion_date, bronze_source_file, _rescued_data
ORDER BY malformed_count DESC
LIMIT 20
"""
                    },
                    {
                        "type": "python",
                        "content": """# Check current schema of bronze table

bronze_table = f"{catalog_name}.{schema_name}.{table_name}"
schema = spark.table(bronze_table).schema

print(f"📊 Current Schema for {bronze_table}:")
print("\nColumns:")
for field in schema.fields:
    nullable = "NULL" if field.nullable else "NOT NULL"
    print(f"  • {field.name:30s} {str(field.dataType):20s} {nullable}")

print(f"\nTotal columns: {len(schema.fields)}")
"""
                    },
                    {
                        "type": "markdown",
                        "content": """## Monitoring & Observability

Key metrics to track:
* **Ingestion lag**: Time between file arrival and ingestion
* **Schema changes**: New columns added over time
* **Error rate**: Percentage of rescued/malformed records
* **Throughput**: Records per minute/hour
* **Checkpoint health**: Checkpoint age and size
"""
                    },
                    {
                        "type": "python",
                        "content": """# Stream monitoring - Get active stream status

active_streams = spark.streams.active

print(f"📡 Active Streams: {len(active_streams)}\n")

for stream in active_streams:
    print(f"Stream ID: {stream.id}")
    print(f"  Name: {stream.name}")
    print(f"  Status: {stream.status}")
    print(f"  Recent Progress:")
    
    if stream.recentProgress:
        latest = stream.recentProgress[-1]
        print(f"    • Batch: {latest.get('batchId', 'N/A')}")
        print(f"    • Input Rows: {latest.get('numInputRows', 'N/A')}")
        print(f"    • Processing Time: {latest.get('durationMs', {}).get('triggerExecution', 'N/A')} ms")
    print()
"""
                    },
                    {
                        "type": "markdown",
                        "content": """## Next Steps

### Silver Layer
Once bronze ingestion is stable, create the silver layer:
* Deduplicate records
* Apply data type conversions
* Join with reference data
* Add business validation rules

### Optimization
* Enable Auto Optimize: `ALTER TABLE ... SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')`
* Configure Z-ORDER for common query patterns
* Set up VACUUM schedule for old files
* Implement retention policies

### Production Checklist
- [ ] Schema evolution alerts configured
- [ ] Checkpoint backup strategy in place
- [ ] Monitoring dashboard created
- [ ] Runbook for common issues documented
- [ ] Access controls configured (Unity Catalog)
- [ ] Data retention policy defined
"""
                    }
                ]
            }
        },
        "best_practices": [
            "Always preserve raw data immutability - never update bronze layer records",
            "Include comprehensive metadata (ingestion timestamp, source file, pipeline ID) for lineage tracking",
            "Enable schema evolution to handle source changes without pipeline failures",
            "Use _rescued_data column to capture malformed records for investigation",
            "Partition by ingestion_date for efficient data lifecycle management",
            "Enable Change Data Feed (CDF) on bronze tables for downstream incremental processing",
            "Implement checkpoint management for exactly-once streaming guarantees",
            "Use Delta Lake's columnMapping.mode for flexible schema evolution",
            "Monitor Auto Loader file notifications to detect ingestion delays",
            "Separate concerns: bronze for ingestion, silver for transformation"
        ],
        "anti_patterns": [
            "❌ Applying business logic transformations in bronze layer",
            "❌ Filtering out 'bad' records - capture everything and flag in silver",
            "❌ Joining multiple sources in bronze - keep sources isolated",
            "❌ Using bronze as a temporary staging area - it's permanent storage",
            "❌ Skipping metadata columns to save storage - they're critical for debugging",
            "❌ Disabling schema evolution in production - embrace it instead"
        ],
        "common_patterns": {
            "pattern_1": {
                "name": "Multi-Source Bronze Consolidation",
                "description": "Ingest data from multiple sources into separate bronze tables with source tracking",
                "code": """# Create separate bronze tables per source with common metadata
@dlt.table(name="bronze_source_a")
def ingest_source_a():
    return add_bronze_metadata(
        spark.readStream.format('cloudFiles').load('/source_a/'),
        source_system='source_a'
    )

@dlt.table(name="bronze_source_b")
def ingest_source_b():
    return add_bronze_metadata(
        spark.readStream.format('cloudFiles').load('/source_b/'),
        source_system='source_b'
    )

def add_bronze_metadata(df, source_system):
    return df.withColumn('source_system', lit(source_system)) \\
             .withColumn('ingestion_timestamp', current_timestamp())
"""
            },
            "pattern_2": {
                "name": "Schema Evolution with Validation",
                "description": "Handle schema changes gracefully while alerting on unexpected evolution",
                "code": """# Monitor schema changes and alert on critical evolution
@dlt.table(name="bronze_with_schema_tracking")
def track_schema_evolution():
    df = spark.readStream.format('cloudFiles') \\
        .option('cloudFiles.schemaLocation', '/schema/location') \\
        .option('cloudFiles.schemaEvolutionMode', 'addNewColumns') \\
        .load('/data/')
    
    # Log schema to monitoring table
    current_schema = df.schema.json()
    # Compare with previous schema and alert if major changes detected
    
    return df
"""
            },
            "pattern_3": {
                "name": "Late Data Handling",
                "description": "Configure watermarks and late data thresholds for time-series data",
                "code": """# Handle late-arriving data with watermarks
@dlt.table(name="bronze_with_watermark")
def handle_late_data():
    return (
        spark.readStream.format('cloudFiles').load('/events/')
        .withWatermark('event_timestamp', '24 hours')  # Allow 24hr late data
        .withColumn('is_late_data', 
                   when(col('event_timestamp') < current_timestamp() - expr('INTERVAL 1 HOUR'), True)
                   .otherwise(False))
    )
"""
            }
        },
        "troubleshooting": {
            "schema_evolution_failures": {
                "symptom": "Pipeline fails with 'schema mismatch' or 'incompatible schema' errors",
                "causes": [
                    "New columns added to source data without compatible types",
                    "Column renamed or removed in source system",
                    "Data type changes (e.g., string to int) not compatible"
                ],
                "solutions": [
                    "Enable cloudFiles.schemaEvolutionMode=addNewColumns for graceful evolution",
                    "Use rescuedDataColumn to capture incompatible records",
                    "Implement schema validation in pre-bronze staging area",
                    "Reset checkpoint and schema location if schema conflict is unresolvable"
                ]
            },
            "checkpoint_issues": {
                "symptom": "Stream reprocesses data or fails to resume from checkpoint",
                "causes": [
                    "Checkpoint location deleted or corrupted",
                    "Checkpoint format incompatible with current Spark version",
                    "Multiple streams writing to same checkpoint location"
                ],
                "solutions": [
                    "Use unique checkpoint locations per stream",
                    "Enable checkpoint location versioning",
                    "Implement checkpoint backup and recovery procedures",
                    "Monitor checkpoint lag metrics"
                ]
            },
            "performance_degradation": {
                "symptom": "Bronze ingestion slows down over time",
                "causes": [
                    "Small file problem - too many small files in source",
                    "Lack of partitioning on bronze tables",
                    "Auto Loader file discovery overhead"
                ],
                "solutions": [
                    "Enable Auto Optimize on bronze tables",
                    "Partition by ingestion_date for efficient pruning",
                    "Configure cloudFiles.maxFilesPerTrigger to control batch size",
                    "Use OPTIMIZE and VACUUM commands regularly"
                ]
            }
        },
        "performance_optimization": [
            "Use Auto Loader instead of readStream.format('parquet') for better scalability",
            "Configure appropriate trigger intervals (e.g., trigger(once=True) for batch, trigger(processingTime='5 minutes') for micro-batch)",
            "Enable Auto Compaction with pipelines.autoOptimize.managed=true",
            "Partition bronze tables by ingestion_date for lifecycle management and query performance",
            "Use Photon acceleration for JSON/CSV parsing performance",
            "Limit concurrent file processing with maxFilesPerTrigger to control resource usage"
        ],
        "sdlc_considerations": {
            "dev": "Test with subset of source files (use maxFilesPerTrigger=10). Mock external sources. Validate schema inference. Test schema evolution scenarios.",
            "test": "Full integration test with production-like data volumes. Validate checkpoint recovery. Test late data handling. Measure ingestion latency and throughput.",
            "prod": "Enable monitoring and alerting on ingestion lag. Implement automated checkpoint backup. Use separate storage for bronze data with retention policies. Enable audit logging."
        },
        "real_world_scenarios": [
            {
                "scenario": "Daily batch ingestion from S3 with schema drift",
                "challenge": "Source system adds new columns without notice, causing pipeline failures",
                "solution": "Enabled Auto Loader with schemaEvolutionMode=addNewColumns and rescuedDataColumn. Implemented schema change detection and alerting. Bronze layer now captures all data regardless of schema changes."
            },
            {
                "scenario": "Real-time event ingestion from Kafka with 10M events/hour",
                "challenge": "Initial implementation had high latency and checkpoint lag",
                "solution": "Optimized with appropriate trigger intervals (5 min micro-batches), enabled Photon, partitioned by ingestion_date, and configured maxFilesPerTrigger. Latency reduced from 30min to 5min."
            }
        ]
    },
    "silver_layer": {
        "name": "Silver Layer Pattern (Cleansed/Conformed)",
        "description": "Data quality, deduplication, conformance and light enrichment. Handles joins and early transformations.",
        "category": "medallion_silver",
        "use_cases": ["Cleansing dirty fields", "Conforming types", "Business rules", "Reference table joins"],
        "parameters": {"join_keys": "list"},
        "code_templates": {"silver": "silver = dlt.read(\"bronze_raw\").dropDuplicates([\"event_id\"])"},
        "best_practices": ["All joins and business rules go here"],
        "sdlc_considerations": {"dev": "Ensure test data covers edge cases"}
    },
    "gold_layer": {
        "name": "Gold Layer Pattern (Curated/Presentation)",
        "description": "Business-level aggregates, reporting tables, SCD dimension handling for analytics.",
        "category": "medallion_gold",
        "use_cases": ["Star schema creation", "Aggregation", "Serving tables for BI"],
        "parameters": {"scd_type": ["SCD1", "SCD2"]},
        "code_templates": {"gold": "gold = silver.groupBy('dim').agg({'val':'sum'})"},
        "best_practices": ["Match business semantic layer"],
        "sdlc_considerations": {"dev": "Coordinate with BI team on schema"}
    },
    # --- Ingestion Patterns ---
    "auto_loader": {
        "name": "Auto Loader Ingestion Pattern",
        "description": "Automated cloud file/stream ingestion supporting schema evolution and scalable change discovery.",
        "category": "ingestion_autoloader",
        "use_cases": ["Incremental file ingest", "Schema drift", "Auto ingestion configuration"],
        "parameters": {"cloud_path": "string", "format": ["json", "csv", "parquet"]},
        "code_templates": {"autoloader": "cloud_files('/mnt/source/', format='json')"},
        "best_practices": ["Configure with schema_location, checkpoint_location"],
        "sdlc_considerations": {"dev": "Use test events, monitor schema inference"}
    },
    "cdc_ingestion": {
        "name": "Change Data Capture (CDC) Ingestion",
        "description": "Apply inserts/updates/deletes from CDC sources for relational/operational data sync.",
        "category": "ingestion_cdc",
        "use_cases": ["Upserts into target table", "SCD Type 1/2 implementation", "Operational data replication"],
        "parameters": {"cdc_table": "string", "target_table": "string", "merge_keys": "list"},
        "code_templates": {"cdc": "MERGE INTO target USING cdc_source ON target.id = cdc_source.id\nWHEN MATCHED AND cdc_source.op = 'D' THEN DELETE\nWHEN MATCHED THEN UPDATE SET *\nWHEN NOT MATCHED THEN INSERT *"},
        "best_practices": ["Track CDC sequence numbers", "Handle deletes properly", "Maintain audit columns"],
        "sdlc_considerations": {"test": "Validate deletes/upserts with small batch"}
    },
    "streaming_ingestion": {
        "name": "Streaming Ingestion Pattern",
        "description": "Real-time streaming data ingestion from Kafka, Event Hubs, Kinesis and other streaming sources.",
        "category": "ingestion_streaming",
        "use_cases": ["Real-time event processing", "Kafka/Event Hubs ingestion", "Low-latency pipelines"],
        "parameters": {"stream_source": "string", "checkpoint_location": "string", "trigger_interval": "string"},
        "code_templates": {"streaming": "df = spark.readStream.format('kafka')\\\n  .option('kafka.bootstrap.servers', 'broker:9092')\\\n  .option('subscribe', 'topic')\\\n  .load()\\\n  .writeStream.format('delta')\\\n  .option('checkpointLocation', '/checkpoint')\\\n  .start()"},
        "best_practices": ["Configure watermarks for late data", "Set appropriate trigger intervals", "Monitor lag metrics"],
        "sdlc_considerations": {"dev": "Use smaller batches, test backfill scenarios"}
    },
    "batch_processing": {
        "name": "Batch Processing Pattern",
        "description": "Traditional batch ingest and transform; process scheduled inbound data in micro-batches.",
        "category": "ingestion_batch",
        "use_cases": ["Nightly ETL load", "Report generation", "Scheduled data processing"],
        "parameters": {"schedule": "cron", "source_table": "string"},
        "code_templates": {"batch": "df = spark.read.table('source')\ndf.write.mode('append').saveAsTable('bronze')"},
        "best_practices": ["Optimize for volume changes", "Implement idempotency", "Add date-based partitioning"],
        "sdlc_considerations": {"prod": "Automate as Lakeflow Job"}
    },
    # --- Jobs and Pipeline Creation ---
    "lakeflow_job_creation": {
        "name": "Lakeflow Job Creation",
        "description": "Author and configure Lakeflow Jobs for recurring, automated workload orchestration (batch/stream).",
        "category": "automation_jobs",
        "use_cases": ["Scheduled pipeline runs", "Multi-task orchestration", "Dependency management"],
        "parameters": {"job_yaml": "string", "tasks": "list"},
        "code_templates": {"job": "# Lakeflow Job YAML example\nname: my_job\ntasks:\n  - notebook_path: /path/to/notebook\n    cluster_key: default\n    depends_on:\n      - task_key: upstream_task"},
        "best_practices": ["Test all job tasks interactively", "Use task dependencies for orchestration", "Configure retry and timeout policies"],
        "sdlc_considerations": {"dev": "Keep in dev folder, mock params"}
    },
    "pipeline_creation": {
        "name": "Lakeflow Pipeline Creation",
        "description": "Define, deploy, and manage Lakeflow Spark Declarative Pipelines (SDP) for ETL/ELT and data transformation design.",
        "category": "automation_pipelines",
        "use_cases": ["Bronze to gold ETL", "Streaming transformations", "Data quality enforcement"],
        "parameters": {"pipeline_yaml": "string", "resources": "dict"},
        "code_templates": {"pipeline": "# SDP YAML config\nresources:\n  pipelines:\n    my_pipeline:\n      name: bronze_to_gold\n      libraries:\n        - notebook:\n            path: /pipelines/bronze_layer.py"},
        "best_practices": ["Modular structure, version control", "Use expectations for data quality", "Separate concerns by layer"],
        "sdlc_considerations": {"prod": "Approval flow required"}
    }
}

print("✓ DE Feature Catalog updated with comprehensive bronze_layer including notebook templates.")

In [0]:
# Create interactive widgets for user input collection

# Widget 1: Select DE feature(s) to generate skill file for
feature_options = list(DE_FEATURE_CATALOG.keys())
display_names = [f"{DE_FEATURE_CATALOG[k]['name']} [{DE_FEATURE_CATALOG[k]['category']}]" for k in feature_options]
dbutils.widgets.dropdown(
    "de_feature",
    feature_options[0],
    feature_options,
    "1. Select DE Feature"
)

# Widget 2: Select target environment
dbutils.widgets.dropdown(
    "environment",
    "dev",
    ["dev", "test", "prod", "all_sdlc"],
    "2. Target Environment"
)

# Widget 3: Target catalog for examples
dbutils.widgets.text(
    "target_catalog",
    "main",
    "3. Target Catalog"
)

# Widget 4: Target schema for examples
dbutils.widgets.text(
    "target_schema",
    "default",
    "4. Target Schema"
)

# Widget 5: Output directory for generated skill files
default_output_path = "/Users/sushant.mishriko@tigeranalytics.com/.assistant/skills"
dbutils.widgets.text(
    "output_path",
    default_output_path,
    "5. Output Directory"
)

# Widget 6: Additional context/requirements
dbutils.widgets.text(
    "additional_context",
    "",
    "6. Additional Context (Optional)"
)

# Widget 7: Include code examples
dbutils.widgets.dropdown(
    "include_code_examples",
    "true",
    ["true", "false"],
    "7. Include Code Examples"
)

# Widget 8: Include best practices
dbutils.widgets.dropdown(
    "include_best_practices",
    "true",
    ["true", "false"],
    "8. Include Best Practices"
)

print("✓ Interactive widgets created successfully!")
print("\n📝 Configure your skill file generation using the widgets above, then run the next cell.")
print(f"\n📦 Available Features ({len(feature_options)}):")
for idx, k in enumerate(feature_options, 1):
    feature_name = DE_FEATURE_CATALOG[k]['name']
    category = DE_FEATURE_CATALOG[k]['category']
    print(f"  {idx:2d}. {feature_name:55s} [{category}]")

In [0]:
# Skill File Template Generator - Converts feature metadata to SKILL.md format

from typing import Dict, Any, List, Tuple
import textwrap
import json

class SkillFileGenerator:
    """
    Generates comprehensive Genie Code skill files in SKILL.md format
    with notebook template support for complex scenarios
    """
    
    def __init__(self, feature_config: Dict[str, Any], user_config: Dict[str, Any]):
        self.feature = feature_config
        self.config = user_config
        self.skill_content = []
    
    def generate(self) -> str:
        """
        Generate complete SKILL.md file content with rich detail
        """
        self._add_header()
        self._add_overview()
        self._add_when_to_use()
        self._add_prerequisites()
        
        # Add notebook templates section BEFORE code examples
        self._add_notebook_templates()
        
        if self.config.get('include_code_examples', 'true') == 'true':
            self._add_code_examples()
        
        self._add_parameters()
        
        if self.config.get('include_best_practices', 'true') == 'true':
            self._add_best_practices()
        
        # Enhanced sections
        self._add_anti_patterns()
        self._add_common_patterns()
        self._add_performance_optimization()
        self._add_sdlc_guidance()
        self._add_troubleshooting()
        self._add_real_world_scenarios()
        self._add_references()
        
        return '\n'.join(self.skill_content)
    
    def get_notebook_template_files(self) -> List[Tuple[str, str]]:
        """
        Extract notebook templates and convert them to .py files for scripts/ directory.
        Returns list of tuples: (filename, content)
        """
        templates = self.feature.get('notebook_templates', {})
        template_files = []
        
        for template_key, template_data in templates.items():
            if isinstance(template_data, dict) and 'cells' in template_data:
                # Convert cells to Python notebook format
                py_content = self._convert_cells_to_py(template_data)
                filename = f"{template_key}.py"
                template_files.append((filename, py_content))
        
        return template_files
    
    def _convert_cells_to_py(self, template_data: Dict[str, Any]) -> str:
        """
        Convert notebook template cells to .py format that can be imported/executed.
        Databricks .py notebooks use special comments to denote cell boundaries.
        """
        lines = []
        lines.append(f"# Databricks notebook source")
        lines.append(f"# MAGIC %md")
        lines.append(f"# MAGIC # {template_data.get('name', 'Notebook Template')}")
        lines.append(f"# MAGIC")
        lines.append(f"# MAGIC {template_data.get('description', '')}")
        lines.append("")
        
        for cell in template_data.get('cells', []):
            cell_type = cell.get('type', 'python')
            content = cell.get('content', '')
            
            lines.append("# COMMAND ----------")
            lines.append("")
            
            if cell_type == 'markdown':
                # Markdown cells
                lines.append("# MAGIC %md")
                for line in content.strip().split('\n'):
                    lines.append(f"# MAGIC {line}")
            elif cell_type == 'sql':
                # SQL cells
                lines.append("# MAGIC %sql")
                for line in content.strip().split('\n'):
                    lines.append(f"# MAGIC {line}")
            else:
                # Python cells (default)
                lines.append(content.strip())
            
            lines.append("")
        
        return '\n'.join(lines)
    
    def _add_header(self):
        """Add skill file header"""
        self.skill_content.extend([
            f"# {self.feature['name']}\n",
            f"**Category:** `{self.feature['category']}`\n",
            f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n",
            f"**Target Environment:** {self.config.get('environment', 'all')}\n",
            f"**Target Catalog:** `{self.config.get('target_catalog', 'main')}.{self.config.get('target_schema', 'default')}`\n",
            "\n---\n"
        ])
    
    def _add_overview(self):
        """Add comprehensive overview section"""
        self.skill_content.extend([
            "## Overview\n",
            f"{self.feature['description']}\n",
            "\n### Purpose\n",
            f"This skill provides **comprehensive, production-ready guidance** for implementing "
            f"{self.feature['name']} in Databricks environments. "
            f"It includes:\n",
            "* Detailed implementation patterns with real code examples\n",
            "* **Complete notebook templates** for rapid development\n",
            "* Best practices learned from production deployments\n",
            "* Common pitfalls and anti-patterns to avoid\n",
            "* Performance optimization techniques\n",
            "* Troubleshooting guides for common issues\n",
            "* SDLC integration strategies (dev/test/prod)\n",
            "* Real-world scenarios and solutions\n",
            "\n"
        ])
    
    def _add_notebook_templates(self):
        """Add notebook templates section"""
        templates = self.feature.get('notebook_templates', {})
        
        if not templates:
            return
        
        self.skill_content.extend([
            "## 📓 Notebook Templates\n",
            "\n### Ready-to-Use Complete Implementations\n\n",
            "**Why Templates Help Genie Code:**\n",
            "* **Faster Development**: Complete, runnable notebooks eliminate the need to piece together code snippets\n",
            "* **Better Context**: Full notebook structure with markdown explanations helps Genie understand complex patterns\n",
            "* **Production Quality**: Templates include monitoring, error handling, and best practices from day one\n",
            "* **Consistency**: Standardized patterns across your organization\n\n",
            "**How to Use Templates:**\n",
            "1. Ask Genie Code to create a notebook using this skill\n",
            "2. Genie will reference the template structure from `scripts/` directory\n",
            "3. Customize the generated notebook for your specific requirements\n\n"
        ])
        
        for template_key, template_data in templates.items():
            if isinstance(template_data, dict):
                template_name = template_data.get('name', template_key)
                template_desc = template_data.get('description', '')
                cells = template_data.get('cells', [])
                
                self.skill_content.extend([
                    f"### Template: {template_name}\n\n",
                    f"**Description:** {template_desc}\n\n",
                    f"**Template File:** `scripts/{template_key}.py`\n\n",
                    f"**Structure:** {len(cells)} cells\n\n",
                    "**Includes:**\n"
                ])
                
                # List what the template includes
                markdown_cells = len([c for c in cells if c.get('type') == 'markdown'])
                python_cells = len([c for c in cells if c.get('type') == 'python'])
                sql_cells = len([c for c in cells if c.get('type') == 'sql'])
                
                self.skill_content.extend([
                    f"* {markdown_cells} documentation cells with explanations\n",
                    f"* {python_cells} Python code cells with complete implementation\n",
                    f"* {sql_cells} SQL cells for monitoring and validation\n",
                    "* Configuration parameters you can customize\n",
                    "* Data quality checks and monitoring queries\n",
                    "* Production-ready error handling\n\n",
                    "**Example Usage with Genie:**\n",
                    "```\n",
                    f"Create a notebook for {template_name.lower()} using the bronze layer template\n",
                    "```\n\n"
                ])
        
        self.skill_content.append("\n")
    
    def _add_when_to_use(self):
        """Add when to use this skill"""
        self.skill_content.extend([
            "## When to Use This Skill\n",
            "\n### Primary Use Cases\n",
            "\nThis skill is essential when you need to:\n\n"
        ])
        
        for idx, use_case in enumerate(self.feature.get('use_cases', []), 1):
            self.skill_content.append(f"{idx}. **{use_case}**\n")
        
        if self.config.get('additional_context'):
            self.skill_content.extend([
                "\n### Additional Context\n",
                f"{self.config['additional_context']}\n"
            ])
        
        self.skill_content.append("\n")
    
    def _add_prerequisites(self):
        """Add prerequisites section"""
        catalog = self.config.get('target_catalog', 'main')
        schema = self.config.get('target_schema', 'default')
        
        self.skill_content.extend([
            "## Prerequisites\n",
            "\n### Required Setup\n",
            "\nBefore implementing this pattern, ensure you have:\n\n",
            "#### Infrastructure\n",
            f"* Access to Databricks workspace with appropriate permissions\n",
            f"* Unity Catalog enabled with `{catalog}` catalog created\n",
            f"* Target schema `{catalog}.{schema}` created and accessible\n",
            "* Compute resources (cluster or SQL warehouse) provisioned and running\n",
            "\n#### Permissions\n",
            f"* `USE CATALOG` on `{catalog}`\n",
            f"* `USE SCHEMA` on `{catalog}.{schema}`\n",
            f"* `CREATE TABLE` and `MODIFY` on `{catalog}.{schema}`\n",
            "* `READ FILES` permission on source data locations\n",
            "\n#### Knowledge\n",
            "* Basic understanding of Databricks notebooks and workspace\n",
            "* Familiarity with PySpark or Databricks SQL\n",
            "* Understanding of Delta Lake fundamentals\n",
            "\n"
        ])
    
    def _add_code_examples(self):
        """Add detailed code examples section"""
        self.skill_content.extend([
            "## Code Examples\n",
            "\n### Production-Ready Implementation Patterns\n",
            "\nThe following examples demonstrate complete, tested implementations "
            "ready for production use.\n\n"
        ])
        
        catalog = self.config.get('target_catalog', 'main')
        schema = self.config.get('target_schema', 'default')
        
        templates = self.feature.get('code_templates', {})
        
        for idx, (template_name, template_code) in enumerate(templates.items(), 1):
            # Detect language from template
            if 'import' in template_code or 'def ' in template_code or 'spark.' in template_code or '@dlt' in template_code:
                lang = 'python'
            elif 'CREATE' in template_code or 'SELECT' in template_code or 'MERGE' in template_code:
                lang = 'sql'
            else:
                lang = 'python'
            
            # Format template name
            display_name = template_name.replace('_', ' ').title()
            
            self.skill_content.extend([
                f"### Example {idx}: {display_name}\n\n"
            ])
            
            # Add description based on template name
            if 'autoloader' in template_name.lower():
                self.skill_content.append(
                    "This example demonstrates Auto Loader configuration with schema evolution, "
                    "metadata tracking, and proper checkpoint management.\n\n"
                )
            elif 'batch' in template_name.lower():
                self.skill_content.append(
                    "This example shows batch ingestion with error handling, partitioning, "
                    "and audit column generation.\n\n"
                )
            elif 'streaming' in template_name.lower() or 'kafka' in template_name.lower():
                self.skill_content.append(
                    "This example demonstrates real-time streaming ingestion with proper "
                    "checkpoint configuration and metadata capture.\n\n"
                )
            elif 'metadata' in template_name.lower():
                self.skill_content.append(
                    "This example shows comprehensive metadata tracking including source lineage, "
                    "audit trail, and pipeline execution context.\n\n"
                )
            
            # Add the code block
            processed_code = template_code.replace('{catalog}', catalog) \
                                         .replace('{schema}', schema) \
                                         .replace('{checkpoint_path}', f'/mnt/checkpoints/{schema}') \
                                         .replace('{source_path}', f'/mnt/data/{schema}/raw') \
                                         .replace('{kafka_brokers}', 'your-kafka-broker:9092') \
                                         .replace('{kafka_topic}', 'your-topic')
            
            self.skill_content.extend([
                f"```{lang}\n",
                processed_code.strip(),
                "\n```\n\n"
            ])
            
            # Add usage notes
            self.skill_content.extend([
                "**Key Configuration Points:**\n",
                "* Adjust paths (`source_path`, `checkpoint_path`) to match your environment\n",
                "* Modify schema and table names as needed\n",
                "* Tune performance parameters based on data volume\n",
                "* Review and customize metadata columns for your use case\n\n"
            ])
    
    def _add_parameters(self):
        """Add comprehensive parameters configuration section"""
        params = self.feature.get('parameters', {})
        
        if not params:
            return
        
        self.skill_content.extend([
            "## Configuration Parameters\n",
            "\n### Parameter Reference\n\n",
            "The following parameters control the behavior of this implementation:\n\n",
            "| Parameter | Type | Description | Example |\n",
            "| --- | --- | --- | --- |\n"
        ])
        
        for param_name, param_desc in params.items():
            # Parse description to extract type and description
            if '-' in str(param_desc):
                parts = str(param_desc).split('-', 1)
                param_type = parts[0].strip()
                description = parts[1].strip()
            elif isinstance(param_desc, list):
                param_type = 'enum'
                description = f"One of: {', '.join(f'`{v}`' for v in param_desc)}"
            else:
                param_type = str(param_desc)
                description = f"Configuration for {param_name}"
            
            # Generate example based on param name
            if 'path' in param_name.lower():
                example = '"/mnt/data/raw"'
            elif 'format' in param_name.lower():
                example = '"parquet"'
            elif 'pattern' in param_name.lower():
                example = '"auto_loader"'
            elif isinstance(param_desc, list):
                example = f'"{param_desc[0]}"'
            else:
                example = '"value"'
            
            self.skill_content.append(
                f"| `{param_name}` | {param_type} | {description} | {example} |\n"
            )
        
        self.skill_content.append("\n")
    
    def _add_best_practices(self):
        """Add comprehensive best practices section"""
        practices = self.feature.get('best_practices', [])
        
        if not practices:
            return
        
        self.skill_content.extend([
            "## Best Practices\n",
            "\n### Recommended Approaches\n\n",
            "Follow these proven best practices for successful implementation:\n\n"
        ])
        
        for idx, practice in enumerate(practices, 1):
            # Add emoji based on practice type
            emoji = "✅"
            if any(word in practice.lower() for word in ['monitor', 'observability', 'alert']):
                emoji = "📊"
            elif any(word in practice.lower() for word in ['security', 'permission', 'access']):
                emoji = "🔒"
            elif any(word in practice.lower() for word in ['performance', 'optimize']):
                emoji = "⚡"
            
            self.skill_content.append(f"{idx}. {emoji} **{practice}**\n")
        
        self.skill_content.append("\n")
    
    def _add_anti_patterns(self):
        """Add anti-patterns section"""
        anti_patterns = self.feature.get('anti_patterns', [])
        
        if not anti_patterns:
            return
        
        self.skill_content.extend([
            "## Anti-Patterns to Avoid\n",
            "\n### Common Mistakes\n\n",
            "Avoid these common pitfalls that lead to issues in production:\n\n"
        ])
        
        for anti_pattern in anti_patterns:
            self.skill_content.append(f"{anti_pattern}\n")
        
        self.skill_content.append("\n")
    
    def _add_common_patterns(self):
        """Add detailed common patterns section"""
        patterns = self.feature.get('common_patterns', {})
        
        if not patterns:
            self.skill_content.extend([
                "## Common Implementation Patterns\n",
                "\n### Pattern 1: Basic Implementation\n",
                f"Start with the simplest implementation of {self.feature['name']} and iterate based on requirements.\n",
                "\n### Pattern 2: Production Deployment\n",
                "Implement proper error handling, logging, monitoring, and recovery mechanisms.\n",
                "\n### Pattern 3: Performance Optimization\n",
                "Apply optimization techniques based on data volume, query patterns, and SLA requirements.\n",
                "\n"
            ])
            return
        
        self.skill_content.extend([
            "## Common Implementation Patterns\n",
            "\n### Proven Patterns for Real-World Scenarios\n\n"
        ])
        
        for pattern_key, pattern_data in patterns.items():
            if isinstance(pattern_data, dict):
                self.skill_content.extend([
                    f"### {pattern_data.get('name', pattern_key)}\n\n",
                    f"**Description:** {pattern_data.get('description', 'Common pattern')}\n\n"
                ])
                
                if 'code' in pattern_data:
                    self.skill_content.extend([
                        "**Implementation:**\n\n",
                        "```python\n",
                        pattern_data['code'].strip(),
                        "\n```\n\n"
                    ])
        
        self.skill_content.append("\n")
    
    def _add_performance_optimization(self):
        """Add performance optimization section"""
        optimizations = self.feature.get('performance_optimization', [])
        
        if not optimizations:
            return
        
        self.skill_content.extend([
            "## Performance Optimization\n",
            "\n### Optimization Techniques\n\n",
            "Apply these techniques to improve performance and reduce costs:\n\n"
        ])
        
        for idx, optimization in enumerate(optimizations, 1):
            self.skill_content.append(f"{idx}. ⚡ {optimization}\n")
        
        self.skill_content.append("\n")
    
    def _add_sdlc_guidance(self):
        """Add comprehensive SDLC-specific guidance"""
        sdlc = self.feature.get('sdlc_considerations', {})
        
        if not sdlc:
            return
        
        self.skill_content.extend([
            "## SDLC Integration\n",
            "\n### Environment-Specific Configuration\n\n"
        ])
        
        env = self.config.get('environment', 'all_sdlc')
        
        env_emojis = {
            'dev': '🛠️',
            'test': '🧪',
            'prod': '🚀'
        }
        
        if env == 'all_sdlc':
            # Show all environments with detailed guidance
            for env_name, guidance in sdlc.items():
                emoji = env_emojis.get(env_name, '📌')
                self.skill_content.extend([
                    f"### {emoji} {env_name.upper()} Environment\n\n",
                    f"{guidance}\n\n"
                ])
        else:
            # Show specific environment
            if env in sdlc:
                emoji = env_emojis.get(env, '📌')
                self.skill_content.extend([
                    f"### {emoji} {env.upper()} Environment\n\n",
                    f"{sdlc[env]}\n\n"
                ])
        
        self.skill_content.append("\n")
    
    def _add_troubleshooting(self):
        """Add comprehensive troubleshooting section"""
        troubleshooting = self.feature.get('troubleshooting', {})
        
        if not troubleshooting:
            # Add generic troubleshooting
            self.skill_content.extend([
                "## Troubleshooting\n",
                "\n### Common Issues\n\n",
                "#### Performance Issues\n",
                "* Check partition strategy and filter predicates\n",
                "* Review execution plans and optimize joins\n",
                "* Consider Z-ORDER optimization for frequently queried columns\n\n",
                "#### Data Quality Problems\n",
                "* Validate expectations and quality rules\n",
                "* Check for schema evolution issues\n",
                "* Review data validation logic\n\n",
                "#### Pipeline Failures\n",
                "* Check error logs and event logs in pipeline monitoring\n",
                "* Verify source data availability and format\n",
                "* Review checkpoint and state management\n\n"
            ])
            return
        
        self.skill_content.extend([
            "## Troubleshooting Guide\n",
            "\n### Common Issues and Solutions\n\n"
        ])
        
        for issue_key, issue_data in troubleshooting.items():
            if isinstance(issue_data, dict):
                self.skill_content.extend([
                    f"### ⚠️ {issue_data.get('symptom', issue_key)}\n\n"
                ])
                
                if 'causes' in issue_data:
                    self.skill_content.append("**Possible Causes:**\n\n")
                    for cause in issue_data['causes']:
                        self.skill_content.append(f"* {cause}\n")
                    self.skill_content.append("\n")
                
                if 'solutions' in issue_data:
                    self.skill_content.append("**Solutions:**\n\n")
                    for idx, solution in enumerate(issue_data['solutions'], 1):
                        self.skill_content.append(f"{idx}. {solution}\n")
                    self.skill_content.append("\n")
        
        self.skill_content.append("\n")
    
    def _add_real_world_scenarios(self):
        """Add real-world scenarios section"""
        scenarios = self.feature.get('real_world_scenarios', [])
        
        if not scenarios:
            return
        
        self.skill_content.extend([
            "## Real-World Scenarios\n",
            "\n### Production Case Studies\n\n",
            "Learn from real implementations and their solutions:\n\n"
        ])
        
        for idx, scenario in enumerate(scenarios, 1):
            if isinstance(scenario, dict):
                self.skill_content.extend([
                    f"### Scenario {idx}: {scenario.get('scenario', 'Case Study')}\n\n",
                    f"**Challenge:** {scenario.get('challenge', 'N/A')}\n\n",
                    f"**Solution:** {scenario.get('solution', 'N/A')}\n\n"
                ])
        
        self.skill_content.append("\n")
    
    def _add_references(self):
        """Add references section"""
        self.skill_content.extend([
            "## Additional Resources\n",
            "\n### Documentation\n\n",
            "* [Databricks Documentation](https://docs.databricks.com)\n",
            "* [Delta Lake Documentation](https://docs.delta.io)\n",
            "* [Unity Catalog Documentation](https://docs.databricks.com/data-governance/unity-catalog)\n",
            "* [Lakeflow Spark Declarative Pipelines](https://docs.databricks.com/workflows/delta-live-tables)\n\n",
            "### Related Skills\n\n",
            "* Check other DE skills for complementary patterns\n",
            "* Review monitoring and observability skills for production deployment\n",
            "* Consult CI/CD deployment skills for automated deployment\n\n",
            "---\n\n",
            f"*Generated by Data Engineering Skill File Generator v2.0 with Notebook Templates*\n",
            f"*Target: {self.config.get('target_catalog', 'main')}.{self.config.get('target_schema', 'default')}*\n"
        ])

print("✓ Enhanced skill file generator engine loaded successfully!")
print("✓ Now generates comprehensive, production-ready SKILL.md files")
print("✓ Includes: code examples, notebook templates, best practices, anti-patterns, troubleshooting")

In [0]:
# Main execution: Generate and save skill file based on user inputs

from pathlib import Path
import os

# Read user inputs from widgets
selected_feature = dbutils.widgets.get('de_feature')
environment = dbutils.widgets.get('environment')
target_catalog = dbutils.widgets.get('target_catalog')
target_schema = dbutils.widgets.get('target_schema')
output_path = dbutils.widgets.get('output_path')
additional_context = dbutils.widgets.get('additional_context')
include_code = dbutils.widgets.get('include_code_examples')
include_practices = dbutils.widgets.get('include_best_practices')

print("=" * 80)
print("DATA ENGINEERING SKILL FILE GENERATOR v2.0")
print("=" * 80)
print(f"\n📋 Configuration:")
print(f"  • Feature: {DE_FEATURE_CATALOG[selected_feature]['name']}")
print(f"  • Environment: {environment}")
print(f"  • Target: {target_catalog}.{target_schema}")
print(f"  • Output: {output_path}")
print(f"  • Code Examples: {include_code}")
print(f"  • Best Practices: {include_practices}")

# Prepare user configuration
user_config = {
    'environment': environment,
    'target_catalog': target_catalog,
    'target_schema': target_schema,
    'additional_context': additional_context,
    'include_code_examples': include_code,
    'include_best_practices': include_practices
}

# Get feature configuration
feature_config = DE_FEATURE_CATALOG[selected_feature]

print(f"\n🔧 Generating skill file...")

# Generate skill file content
generator = SkillFileGenerator(feature_config, user_config)
skill_content = generator.generate()

# Create output directory if it doesn't exist
output_dir = Path(output_path) / selected_feature
output_dir_str = str(output_dir)

try:
    dbutils.fs.mkdirs(f"/Workspace{output_dir_str}")
    print(f"✓ Created directory: {output_dir_str}")
except Exception as e:
    print(f"ℹ Directory already exists or created: {output_dir_str}")

# Save SKILL.md file
skill_file_path = output_dir / "SKILL.md"
skill_file_path_str = str(skill_file_path)

try:
    # Write SKILL.md to workspace using dbutils.fs.put
    dbutils.fs.put(f"/Workspace{skill_file_path_str}", skill_content, overwrite=True)
    
    print(f"\n✅ Skill file generated successfully!")
    print(f"📁 Location: {skill_file_path_str}")
    print(f"📊 Size: {len(skill_content)} characters")
    
    # Generate and save notebook templates
    template_files = generator.get_notebook_template_files()
    
    if template_files:
        print(f"\n📓 Generating notebook templates...")
        
        # Create scripts subdirectory
        scripts_dir = output_dir / "scripts"
        scripts_dir_str = str(scripts_dir)
        
        try:
            dbutils.fs.mkdirs(f"/Workspace{scripts_dir_str}")
            print(f"✓ Created scripts directory: {scripts_dir_str}")
        except Exception as e:
            print(f"ℹ Scripts directory exists: {scripts_dir_str}")
        
        # Save each template file
        for filename, content in template_files:
            template_file_path = scripts_dir / filename
            template_file_path_str = str(template_file_path)
            
            dbutils.fs.put(f"/Workspace{template_file_path_str}", content, overwrite=True)
            print(f"  ✓ Created template: {filename} ({len(content)} characters)")
        
        print(f"\n✅ Generated {len(template_files)} notebook template(s)")
    else:
        print(f"\nℹ No notebook templates defined for this feature")
    
    # Generate summary
    print(f"\n📊 Generation Summary:")
    print(f"  • Feature Name: {feature_config['name']}")
    print(f"  • Category: {feature_config['category']}")
    print(f"  • Use Cases: {len(feature_config.get('use_cases', []))}")
    print(f"  • Code Examples: {len(feature_config.get('code_templates', {}))}")
    print(f"  • Notebook Templates: {len(template_files)}")
    print(f"  • Best Practices: {len(feature_config.get('best_practices', []))}")
    print(f"  • Troubleshooting Items: {len(feature_config.get('troubleshooting', {}))}")
    print(f"  • Real-World Scenarios: {len(feature_config.get('real_world_scenarios', []))}")
    
    # Create a preview
    preview_lines = skill_content.split('\n')[:60]
    preview = '\n'.join(preview_lines)
    
    print(f"\n" + "=" * 80)
    print("SKILL.md PREVIEW (First 60 lines)")
    print("=" * 80)
    print(preview)
    print("\n... (content continues) ...\n")
    
    print("\n" + "=" * 80)
    print("✅ GENERATION COMPLETE")
    print("=" * 80)
    print(f"\n📝 Next Steps:")
    print(f"  1. Review the generated SKILL.md at: {skill_file_path_str}")
    if template_files:
        print(f"  2. Check notebook templates in: {scripts_dir_str}")
        print(f"  3. Test templates with Genie Code: Ask 'Create a notebook using {selected_feature} skill'")
        print(f"  4. Customize templates for your organization's patterns")
    else:
        print(f"  2. Customize code examples for your specific use case")
        print(f"  3. Test the skill file with Genie Code")
    print(f"  5. Iterate and refine based on team feedback")
    print(f"\n🎯 To generate another skill file, update the widgets above and re-run this cell.")
    
except Exception as e:
    print(f"\n❌ Error generating skill file: {str(e)}")
    print(f"\nTroubleshooting:")
    print(f"  • Verify output path has write permissions")
    print(f"  • Check that the directory path is valid")
    print(f"  • Ensure no special characters in file names")
    import traceback
    print(f"\n🔍 Full error trace:")
    traceback.print_exc()
    raise